In [1]:
!cp -r /kaggle/input/datasets/sangrampatil5150/unet-rogii/* /kaggle/working/

In [2]:
import os
import gc
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import joblib
import torch
from scipy.signal import savgol_filter

# Modular custom imports (assumes these files exist in your working directory)
import utils
from model import SCA_U2Net_Reg, UNetRegConfig
from train import predict_test_well, build_test_features

warnings.filterwarnings("ignore")

# =============================================================================
# Inference Configurations
# =============================================================================

class CFG:
    dataset_path = Path("/kaggle/input/competitions/rogii-wellbore-geology-prediction")
    artifacts_path = Path("/kaggle/working/")

# Standard sequence sliding configuration
WINDOW_SIZE = 512
STRIDE = 256
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# Paths to your saved training output
NORM_FILE = CFG.artifacts_path / "sca_u2net_norm_stats.pkl"
MODEL_DIR = CFG.artifacts_path

def run_standalone_inference():
    print("=" * 70)
    print(" SCA-U2Net 5-Fold Ensemble Standalone Inference")
    print("=" * 70)

    # 1. Load your pre-fit global normalization statistics
    if not NORM_FILE.exists():
        raise FileNotFoundError(f"Normalization stats file not found at {NORM_FILE}. Did step 2 finish successfully?")
    
    norm_data = joblib.load(NORM_FILE)
    norm_mean, norm_std = norm_data['mean'], norm_data['std']
    print(f"  ✓ Loaded global normalization stats from {NORM_FILE}")

    # 2. Instantiate and load all 5 trained fold model weights
    models = []
    unet_cfg = UNetRegConfig()
    
    for fold in range(5):
        ckpt_path = MODEL_DIR / f"sca_u2net_fold{fold}.pt"
        if not ckpt_path.exists():
            print(f"  [WARN] Checkpoint for Fold {fold} not found at {ckpt_path}. Skipping.")
            continue
            
        model = SCA_U2Net_Reg(unet_cfg)
        model.load_state_dict(torch.load(ckpt_path, map_location="cpu", weights_only=True))
        model.eval()
        models.append(model)
        
    if len(models) == 0:
        raise ValueError(f"No trained models loaded from {MODEL_DIR}. Please check your checkpoint files.")
    print(f"  ✓ Loaded {len(models)} trained SCA-U2Net fold models into memory.")

    # 3. Process and build features for test wells
    train_dir = CFG.dataset_path / "train"
    test_dir = CFG.dataset_path / "test"
    
    print("\n  Building test well features...")
    test_well_data = build_test_features(test_dir, train_dir)
    if not test_well_data:
        raise ValueError("No test wells processed. Check if the test data paths are correct.")

    # 4. Run sliding-window inference with model ensembling
    print(f"\n  Running 5-fold ensemble inference across {len(test_well_data)} test wells...")
    test_predictions = []
    
    for wd in test_well_data:
        # predict_test_well handles: averaging 5 fold models, moving them to GPU/CPU safely,
        # and applying Savitzky-Golay trajectory smoothing to the predictions.
        pred_tvt, pred_delta = predict_test_well(
            models, wd, norm_mean, norm_std, DEVICE, 
            window_size=WINDOW_SIZE, stride=STRIDE, apply_sg=True
        )
        test_predictions.append(pred_tvt)
        print(f"    ✓ {wd['wid']}: N={len(pred_tvt)} points, TVT range=[{pred_tvt.min():.2f}, {pred_tvt.max():.2f}]")

    # 5. Compile and generate submission file matching the sample schema
    print("\n  Assembling submission CSV...")
    rows = []
    for wd, pred_tvt in zip(test_well_data, test_predictions):
        wid = wd['wid']
        ev_indices = wd['ev_indices']
        for i, row_idx in enumerate(ev_indices):
            rows.append({
                'id': f"{wid}_{row_idx}",
                'tvt': float(pred_tvt[i])
            })
            
    sub_df = pd.DataFrame(rows)

    # Re-align with Kaggle's sample submission to guarantee correct index order
    sample_path = CFG.dataset_path / "sample_submission.csv"
    if sample_path.exists():
        sample = pd.read_csv(sample_path)
        sub_df = sample[['id']].merge(sub_df, on='id', how='left')
        
        # Safe fallback: fill any potential NaN anomalies with the overall median
        missing_count = sub_df['tvt'].isna().sum()
        if missing_count > 0:
            print(f"  [WARN] {missing_count} rows missing predictions. Filling with median.")
            sub_df['tvt'] = sub_df['tvt'].fillna(sub_df['tvt'].median())
    else:
        print("  [INFO] sample_submission.csv not found in input directory. Outputting raw predicted rows.")

    output_path = "submission.csv"
    sub_df[['id', 'tvt']].to_csv(output_path, index=False)
    print(sub_df)
    print(f"✓ Standalone Inference Complete! Submission saved to {output_path} ({len(sub_df)} rows).")

if __name__ == "__main__":
    run_standalone_inference()

 SCA-U2Net 5-Fold Ensemble Standalone Inference
  ✓ Loaded global normalization stats from /kaggle/working/sca_u2net_norm_stats.pkl
  ✓ Loaded 5 trained SCA-U2Net fold models into memory.

  Building test well features...
  Building spatial imputers (773 wells)...

  Running 5-fold ensemble inference across 3 test wells...
    ✓ 000d7d20: N=3836 points, TVT range=[11735.67, 11748.96]
    ✓ 00bbac68: N=6014 points, TVT range=[12198.91, 12239.67]
    ✓ 00e12e8b: N=4301 points, TVT range=[11598.45, 11612.19]

  Assembling submission CSV...
                  id           tvt
0      000d7d20_1442  11747.114258
1      000d7d20_1443  11747.116211
2      000d7d20_1444  11747.118164
3      000d7d20_1445  11747.120117
4      000d7d20_1446  11747.122070
...              ...           ...
14146  00e12e8b_6379  11604.388672
14147  00e12e8b_6380  11604.341797
14148  00e12e8b_6381  11604.290039
14149  00e12e8b_6382  11604.232422
14150  00e12e8b_6383  11604.167969

[14151 rows x 2 columns]
✓ Standalon